## 13-feature-era.ipynb

Builds the `era_bin` feature and sparse era matrix for every album in the `valid_albums` scope
using an Option D fallback chain across three year sources:

1. `release_group_meta_year` — canonical MusicBrainz first-release year (`mb_release_year.parquet`)
2. `album_country_year` — earliest country-specific release date (`mb_album_country.parquet`)
3. `artist_begin_year` — band formation / artist birth year (`mb_artist.parquet`)

Five known data-entry errors (verified via web search) are corrected before binning.
Years > 2026 are hard-capped to `Unknown`.

**Era bins:** `Pre-1900`, `1900–1949` (50-year buckets for sparse pre-modern data),
then `1950s` → `2020s` in standard decade bins. `Unknown` albums get an all-zero row in
the matrix — no signal, no column.

**Data review:** see `1-EDA/07-EDA-year.ipynb` for coverage analysis, source agreement,
outlier review, and sanity checks.

**Inputs:** `data/mb_release_year.parquet`, `data/mb_album_country.parquet`,
`data/mb_album_artists.parquet`, `data/mb_artist.parquet`, `data/mb_album.parquet`

**Outputs:**
- `data/features/album_era.parquet` — one row per album: `album_id`, `best_year`, `best_year_source`, `era_bin`
- `data/features/album_era_matrix.npz` — sparse one-hot matrix `(n_albums × 12)` aligned to `album_ids.pkl`

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

## Load sources

Loads the four input files. Each provides a different year signal:

- **`mb_release_year.parquet`** — one row per album with a known `release_group_meta_year`.
  Only albums where `release_group_meta.first_release_date_year IS NOT NULL` are included;
  albums with no year in this table are absent (not a zero row), handled downstream as NULL.
  Exported by the new cell in `01-postgres-to-parquet.ipynb`.
- **`mb_album_country.parquet`** — one row per album (earliest country release selected at
  import time). `album_year` renamed to `album_country_year` to avoid name collisions.
- **`mb_album_artists.parquet`** — primary artist per album. Used purely as a join bridge to
  get `artist_name` (for display) and `artist_id` (to look up the artist's begin year).
- **`mb_artist.parquet`** — full artist table. Only `id` and `artist_year` (= `begin_date_year`)
  are loaded; all other artist columns are irrelevant here.

In [ ]:
albums = pd.read_parquet(f'{DATA_DIR}/mb_album.parquet').rename(columns={'id': 'album_id'})

rg_year = pd.read_parquet(f'{DATA_DIR}/mb_release_year.parquet')

country_year = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_country.parquet', columns=['album_id', 'album_year'])
    .rename(columns={'album_year': 'album_country_year'})
)

album_artists = pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet', columns=['album_id', 'artist_id', 'artist_name'])
artists = pd.read_parquet(f'{DATA_DIR}/mb_artist.parquet', columns=['id', 'artist_year']).rename(columns={'id': 'artist_id', 'artist_year': 'artist_begin_year'})
artist_year = album_artists.merge(artists, on='artist_id', how='left')[['album_id', 'artist_name', 'artist_begin_year']]

print(f'Albums in scope : {len(albums):,}')
print(f'rg_meta rows    : {len(rg_year):,}')
print(f'country rows    : {len(country_year):,}')
print(f'artist rows     : {len(artist_year):,}')

## Assemble: join all three sources

Left-joins all three year sources onto the master album list so every album in scope appears
in the result, even if all three year columns are NULL. The join order matters:

1. Start from `mb_album.parquet` — this is the scope boundary. Every album here is in
   `valid_albums` and will have a row in the final feature matrix.
2. Left-join `rg_year` — adds `release_group_meta_year`. Albums absent from `mb_release_year`
   get NULL (they had no year in `release_group_meta`).
3. Left-join `country_year` — adds `album_country_year`. Albums with no country release event
   get NULL.
4. Left-join `artist_year` — adds `artist_name` and `artist_begin_year`. Albums with no
   linked artist (VA releases) or artists with no formation year get NULL.

All three source columns are kept separate so the fallback chain below can apply them in
priority order and track which source was used.

In [ ]:
df = (
    albums[['album_id', 'name']]
    .merge(rg_year, on='album_id', how='left')
    .merge(country_year, on='album_id', how='left')
    .merge(artist_year, on='album_id', how='left')
)
print(f'Assembled: {len(df):,} rows')

## Known year corrections

Five albums with confirmed bad years, verified via web search during EDA (see `07-EDA-year.ipynb`).

The corrections are stored as a dict and applied **after** `best_year` is derived from the
fallback chain, not to the raw source columns. This means:
- The original source columns (`release_group_meta_year`, `album_country_year`,
  `artist_begin_year`) are untouched and remain auditable.
- `best_year_source` is set to `'manual_correction'` for patched rows so they can be
  identified downstream.

The table below documents the evidence for each correction. All five were confirmed via
Bandcamp, Discogs, streaming services, or the artist's label website.

| album_id | Album | Artist | Raw year | Corrected | Source |
|---|---|---|---|---|---|
| 4576816 | Signes & Racines | Adag'nan | 2027 | 2021 | Bandcamp / Amazon |
| 4598782 | Original Music Box Melodies Of Christmas | Christmas Music Box | 2069 | 1969 | Discogs (Pickwick vinyl) |
| 4615228 | Meditations, Vol. 1 | Helisir | 2205 | 2025 | Bandcamp |
| 4643359 | diSTILLed | Russ Still | 2202 | 2022 | Artist confirmed real; transposition |
| 4706113 | The Entertainment | The Clockworks | 2027 | 2026 | V2 Records (March 2026) |

In [ ]:
YEAR_CORRECTIONS = {
    4576816: 2021,
    4598782: 1969,
    4615228: 2025,
    4643359: 2022,
    4706113: 2026,
}

## Option D fallback chain + era binning

**Fallback chain** (`pandas.Series.combine_first` applies the priority order):

```
best_year = release_group_meta_year
         ?? album_country_year
         ?? artist_begin_year
```

`best_year_source` records which source won for each album — useful for auditing the quality
of era assignments downstream (e.g. era bins sourced from `artist_begin` are lower-confidence
than those sourced from `release_group_meta`).

**`assign_era()` binning rules:**
- `NULL` → `'Unknown'` (no year in any source)
- `> 2026` → `'Unknown'` (hard cap for future/erroneous dates; seven unverifiable 2027 entries
  remain after corrections and are caught here)
- `< 1900` → `'Pre-1900'` (50-yr bucket — very sparse)
- `1900–1949` → `'1900–1949'` (50-yr bucket — sparse pre-modern data)
- `1950+` → `'NNNNs'` where NNNN = `(year // 10) * 10` (e.g. 1973 → `'1970s'`)

**Why `float(corrected_year)` in the correction loop?** `best_year` is a float64 Series
(because it may contain NaN values from the combine_first chain). Assigning an int would
silently upcast or raise; casting explicitly keeps the dtype consistent.

In [ ]:
df['best_year'] = (
    df['release_group_meta_year']
    .combine_first(df['album_country_year'])
    .combine_first(df['artist_begin_year'])
)
df['best_year_source'] = np.select(
    [
        df['release_group_meta_year'].notna(),
        df['album_country_year'].notna(),
        df['artist_begin_year'].notna(),
    ],
    ['release_group_meta', 'album_country', 'artist_begin'],
    default='unknown'
)

for album_id, corrected_year in YEAR_CORRECTIONS.items():
    mask = df['album_id'] == album_id
    old  = df.loc[mask, 'best_year'].values
    df.loc[mask, 'best_year']        = float(corrected_year)
    df.loc[mask, 'best_year_source'] = 'manual_correction'
    print(f'  album_id={album_id}  {old[0]:.0f} → {corrected_year}')

def assign_era(year):
    if pd.isna(year): return 'Unknown'
    y = int(year)
    if y > 2026: return 'Unknown'
    if y < 1900: return 'Pre-1900'
    if y < 1950: return '1900–1949'
    return f'{(y // 10) * 10}s'

df['era_bin'] = df['best_year'].apply(assign_era)

print(f'\nSource breakdown:')
print(df['best_year_source'].value_counts().to_string())
print(f'\nEra bin counts:')
era_order = ['Pre-1900', '1900–1949'] + [f'{d}s' for d in range(1950, 2030, 10)] + ['Unknown']
print(df['era_bin'].value_counts().reindex(era_order, fill_value=0).to_string())

## Save feature parquet

Saves the per-album year and era columns to `data/features/album_era.parquet`. This parquet
is the input to the matrix-building step below and is also useful independently:

- `album_id` — join key back to any other feature table
- `best_year` — raw year value (float; NULL for Unknown-era albums)
- `best_year_source` — which source provided the year (`release_group_meta`, `album_country`,
  `artist_begin`, `manual_correction`, or `unknown`)
- `era_bin` — string era label used as the matrix column name

The parquet is saved separately from the matrix so that downstream code (e.g. the app) can
read era labels without loading the full sparse matrix.

In [ ]:
out = df[['album_id', 'best_year', 'best_year_source', 'era_bin']]
out.to_parquet(f'{FEATURES_DIR}/album_era.parquet', index=False, compression='zstd')

print(f'Saved: {FEATURES_DIR}/album_era.parquet')
print(f'Shape : {out.shape}')
print(f'Unknown era: {(out["era_bin"] == "Unknown").sum():,}  ({(out["era_bin"] == "Unknown").mean()*100:.1f}%)')

## Build era matrix

Constructs a sparse one-hot matrix aligned to the master `album_ids.pkl` index, following the
same contract as all other feature matrices in the app:

- **Rows** = albums, in the exact order of `album_ids.pkl` (1,758,488 rows)
- **Columns** = era bins in chronological order (`Pre-1900`, `1900–1949`, `1950s`, … `2020s`)
- **Values** = 1.0 (float32) where an album belongs to that era; 0.0 everywhere else

**Why `Unknown` has no column:** Albums with no year have an all-zero row, contributing zero
to any cosine similarity. Giving `Unknown` its own column would make two no-data albums appear
similar to each other — a false signal. Zero rows are the correct representation of
"no era information".

**Why float32?** All other feature matrices use float32. Consistent dtype avoids implicit
upcasting when the weighted-cosine code sums across blocks.

**`get_indexer` alignment:** Maps album IDs and era labels to row/column positions in the
master index. Returns -1 for any ID not in the index (shouldn't happen, but the `valid`
mask filters those out defensively). This is the same pattern used in `album_country_matrix`
and `album_genre_matrix`.

In [ ]:
import pickle
from scipy.sparse import csr_matrix, save_npz

# Load master album index — all matrices must align to this
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums = len(album_index)
print(f'Master album universe: {n_albums:,}')

In [ ]:
era_parquet = pd.read_parquet(f'{FEATURES_DIR}/album_era.parquet')

# Ordered era vocabulary — Unknown excluded (zero row, not a column)
ERA_ORDER = ['Pre-1900', '1900–1949'] + [f'{d}s' for d in range(1950, 2030, 10)]
era_index = pd.Index(ERA_ORDER)
n_eras = len(era_index)

# Only keep albums with a known era and present in the master index
era_known = era_parquet[era_parquet['era_bin'] != 'Unknown'].copy()

row_idx = album_index.get_indexer(era_known['album_id'].values)
col_idx = era_index.get_indexer(era_known['era_bin'].values)

valid = (row_idx >= 0) & (col_idx >= 0)

X_era = csr_matrix(
    (np.ones(valid.sum(), dtype=np.float32),
     (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_eras)
)

print(f'Era columns    : {ERA_ORDER}')
print(f'X_era shape    : {X_era.shape}')
print(f'Non-zero rows  : {(X_era.sum(axis=1) > 0).sum():,}  ({(X_era.sum(axis=1) > 0).mean()*100:.1f}%)')
print(f'Zero rows (Unknown era): {(X_era.sum(axis=1) == 0).sum():,}')

## Spot-check: column sums

Confirms the per-column album counts in the matrix match the era distribution produced by
`assign_era()`. If these numbers differ from the `era_bin` value counts printed in the
era-bins cell, there is a mismatch in the `get_indexer` alignment (e.g. an album ID present
in the parquet but absent from `album_ids.pkl`, which would be dropped by the `valid` mask).

In [ ]:
col_sums = np.asarray(X_era.sum(axis=0)).flatten()
print('Albums per era column:')
for era, count in zip(ERA_ORDER, col_sums):
    print(f'  {era:<12} {int(count):>8,}')

## Save matrix

Saves the sparse matrix to `data/features/album_era_matrix.npz` in scipy's compressed sparse
format. This file is loaded by `app_v3_weighted.py` as one of the six feature blocks, weighted
by the ERA knob (default dial 4 = weight 0.73).

The matrix is small (~3 MB) because it is maximally sparse — each row has exactly one non-zero
entry (or zero entries for Unknown-era albums).

In [ ]:
save_npz(f'{FEATURES_DIR}/album_era_matrix.npz', X_era)
print(f'Saved : {FEATURES_DIR}/album_era_matrix.npz')
print(f'Shape : {X_era.shape}')
print(f'nnz   : {X_era.nnz:,}')